# Initial state prior shapes

`phi_init` is a Dirichlet over `phi_init_bins` bins. Those bins are expanded to the
`n_states - 2*response_width` free states, then padded with zeros into the response zones.

    phi_init (n_bins)  ->  expanded (n_free)  ->  phi_0 (n_states)

The concentration is fixed and model specific: a centre bump for Markov, bumps at both
boundaries for Quantum. It sets the prior *mean*; the data moves `phi_init` away from it.

Binning exists because the initial state is only weakly identified. With 30 trials per
participant, `n_free - 1` free dimensions is far more than the data can resolve, which showed
up as `phi_init` r_hat around 1.2-1.85. `n_bins - 1` dimensions is what the data can support.

This notebook imports the model's own functions, so it always matches what the model does.

In [ ]:
import numpy as np
import jax
import jax.numpy as npx
import matplotlib.pyplot as plt
import numpyro.distributions as dist
import cme.decision_models.confidence_accumulation as ca

N_STATES = 51
RESPONSE_WIDTH = 17
N_BINS = 5            # phi_init_bins; None in the model means round(0.1 * n_states)

N_FREE = N_STATES - 2 * RESPONSE_WIDTH
rng = jax.random.PRNGKey(0)

print(f"n_states={N_STATES}  response_width={RESPONSE_WIDTH}  ->  n_free={N_FREE}")
print(f"n_bins={N_BINS}  ->  {N_BINS - 1} free dimensions per participant "
      f"(was {N_FREE - 1})")

## The concentration

`_initial_state_concentration` is called with `n_bins`, so the bump shape appears at bin
resolution. `PHI_CONC_AMP` controls how pronounced it is; `sum(conc)` is roughly how many
pseudo-observations the prior is worth against your 30 real trials.

In [ ]:
conc = {mt: np.asarray(ca._initial_state_concentration(N_BINS, mt)) for mt in ["Markov", "Quantum"]}

for mt, c in conc.items():
    print(f"{mt:8s} conc      = {c.round(2)}")
    print(f"{'':8s} prior mean% = {(100*c/c.sum()).round(1)}")
    print(f"{'':8s} sum(conc)  = {c.sum():.1f}   (vs 30 trials)")
    print()

In [ ]:
bins = np.arange(N_BINS)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (mt, c) in zip(axes, conc.items()):
    ax.bar(bins, 100 * c / c.sum(), color="C0")
    ax.set_title(f"{mt}: prior mean over bins", fontsize=10)
    ax.set_xlabel("bin")
    ax.set_xticks(bins)
axes[0].set_ylabel("prior mean E[p]  (%)")
plt.tight_layout()

## Bins expanded to states

`_expand_bins` spreads each bin's mass evenly over the states it covers, dividing by the bin
size so the total stays 1. Bins are as equal as `n_free` allows - 17 states into 5 bins gives
sizes `[4, 4, 3, 3, 3]`. The result is a staircase; the diffusion smooths it within a few
timesteps.

In [ ]:
sizes = [len(s) for s in np.array_split(np.arange(N_FREE), N_BINS)]
print(f"{N_FREE} states into {N_BINS} bins -> sizes {sizes}")

n_draws = 8
states = np.arange(N_FREE) - (N_FREE - 1) / 2

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, (mt, c) in zip(axes, conc.items()):
    draws = dist.Dirichlet(npx.asarray(c)).sample(rng, (n_draws,))
    for d in draws:
        ax.step(states, 100 * np.asarray(ca._expand_bins(d, N_FREE, N_BINS)),
                where="mid", c="grey", alpha=0.45, lw=1)
    mean = ca._expand_bins(npx.asarray(c / c.sum()), N_FREE, N_BINS)
    ax.step(states, 100 * np.asarray(mean), where="mid", c="C3", lw=2.5, label="prior mean")
    ax.set_title(f"{mt}: expanded initial state, {n_draws} draws", fontsize=10)
    ax.set_xlabel("free state (centred)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("p  (%)")
plt.tight_layout()

## Choosing n_bins

Fewer bins means fewer free dimensions and more trials per parameter, but a coarser shape.
More bins also raises `sum(conc)`, so the prior carries more weight relative to the data.

In [ ]:
n_trials = 30
print(f"{'n_bins':>7} {'free dims':>10} {'trials/param':>13} {'sum(conc)':>11} {'prior vs data':>14}")
for k in [3, 5, 7, 9, 11]:
    c = np.asarray(ca._initial_state_concentration(k, "Markov"))
    print(f"{k:>7} {k-1:>10} {n_trials/(k-1):>13.1f} {c.sum():>11.1f} {n_trials/c.sum():>13.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, k in zip(axes, [3, 5, 9]):
    c = npx.asarray(ca._initial_state_concentration(k, "Markov"))
    for d in dist.Dirichlet(c).sample(rng, (n_draws,)):
        ax.step(states, 100 * np.asarray(ca._expand_bins(d, N_FREE, k)),
                where="mid", c="grey", alpha=0.45, lw=1)
    ax.step(states, 100 * np.asarray(ca._expand_bins(c / c.sum(), N_FREE, k)),
            where="mid", c="C3", lw=2.5)
    ax.set_title(f"Markov, n_bins={k}  ({k-1} free dims)", fontsize=10)
    ax.set_xlabel("free state")
axes[0].set_ylabel("p  (%)")
plt.tight_layout()

## In the full state space

The expanded initial state is padded with `response_width` zeros on each side to give `phi_0`. For Quantum
`phi_0 = sqrt(p_0)`, since it is an amplitude rather than a probability. Note where the
Quantum boundary mass sits relative to the response zones.

In [ ]:
full = np.arange(N_STATES) - (N_STATES - 1) / 2
pad = (RESPONSE_WIDTH, RESPONSE_WIDTH)

fig, ax = plt.subplots(figsize=(9, 3.4))
for mt, c in conc.items():
    mean = np.asarray(ca._expand_bins(npx.asarray(c / c.sum()), N_FREE, N_BINS))
    p_full = np.pad(mean, pad)
    y = p_full if mt == "Markov" else np.sqrt(p_full)
    ax.step(full, y, where="mid", lw=2,
            label=f"{mt}  phi_0 = {'p_0' if mt == 'Markov' else 'sqrt(p_0)'}")

ax.axvspan(full[0], full[RESPONSE_WIDTH - 1], color="grey", alpha=0.18)
ax.axvspan(full[-RESPONSE_WIDTH], full[-1], color="grey", alpha=0.18)
ax.text(full[RESPONSE_WIDTH // 2], 0.02, "response\nzone", ha="center", fontsize=8, c="grey")
ax.set_xlabel("confidence state")
ax.set_ylabel("phi_0")
ax.set_title("Initial state across all states (shaded = response zones)")
ax.legend(fontsize=8)
plt.tight_layout()

## Sanity check against the model

Traces `_get_initial_state` itself, so this fails if the notebook and the model disagree.

In [ ]:
from numpyro import handlers

for mt in ["Markov", "Quantum"]:
    def site():
        return ca._get_initial_state(N_STATES, N_FREE // 2, RESPONSE_WIDTH, I=2, prob=1,
                                     model_type=mt, prior_type="Model", phi_init_bins=N_BINS)
    tr = handlers.trace(handlers.seed(site, rng)).get_trace()
    shapes = {k: tuple(np.asarray(tr[k]["value"]).shape) for k in ["phi_conc", "phi_init", "phi_0"]}
    total = float(np.asarray(tr["phi_init"]["value"])[0, 0, 0].sum())
    print(f"{mt:8s} {shapes}  phi_init sums to {total:.4f}")